In [3]:
print(R.version$version.string)

[1] "R version 4.6.1 (2026-06-24 ucrt)"


In [74]:
suppressPackageStartupMessages({
    library("ape")
    library("phytools")
    library("corHMM")
    library("OUwie")
    library("readxl")
    library("ggplot2")
})
source("../houwie_patches.R") # this contains patched versions of OUwie::getModelTable and OUwie::getModelAvgParams

read_all_rds <- function(dirpath, cd, strip, rm_underscores = TRUE) {
    # dirpath = path to the directory containing the .Rds files
    # cd = character dependent or independent models, these are expected to contain "CD" or "CID" in their names
    # strip = the regex pattern to strip out of the file names when naming objects, can be a plain string as well
    # rm_underscores - strip all the underscores in the object name

    fnames <- list.files(dirpath) # all the files in the specified dir
    if(cd) fnames <- fnames[grep(pattern = "CD", fnames)]
    else fnames <- fnames[grep(pattern = "CID", fnames)] # cherry pick CD or CID models, given their names contain "CD" or "CID" to distinguish them
    # print(fnames)

    paths <- paste0(dirpath, fnames) # relative paths for all the Rds files
    stopifnot(length(fnames)==length(paths))

    # remove the unnecessary parts of the file names to create the model names and also remove all the underscores
    mnames <- fnames |> gsub(pattern = ".Rds", replacement = '') |> gsub(pattern = strip, replacement = '') |> gsub(pattern = ifelse(rm_underscores, '_', ''), replacement = '')
    stopifnot(length(fnames)==length(mnames))
    stopifnot(length(paths)==length(mnames))

    models <- list()
    for (i in seq_along(mnames)) models[[mnames[i]]] <- readRDS(file = paths[i])
    stopifnot(length(models)==length(mnames))

    models
}

In [13]:
packageVersion("OUwie")

[1] '3.0.3'

In [14]:
packageVersion("corHMM")

[1] '2.8'

In [42]:
fred4 <- read.csv("../../data/chapter2/FRED/subsets/continuous_raw.csv")
s <- readxl::read_xlsx("../../data/chapter2/FRED/subsets/final.xlsx", sheet = "final")[c("binominal", "state")]
s$binominal <- gsub(s$binominal, pattern = ' ', replacement = '_')
fred4 <- merge(fred4, s, by = "binominal", all.x = TRUE)

# ___Hypotheses___

- Evolution of mycorrhizal states and fine root ecological strategies will be correlated. We expect that the RD, SRL and RTD will show correlated evolution with mycorrhizal states (__overall character dependency - if found show what OU parameters show character dependency__).
  
- Evolutionary rates of the fine root traits $\sigma^2$ would be higher when they are paired to a hybrid (dual) mycorrhizal state (e.g. AM/NM, AM/EcM) than a singular mycorrhizal state (e.g. AM, EcM). 
  
- Evolutionary rate $\sigma^2$ of fine root traits should be the lowest when paired to NM.

__Can also add the pull towards the optima $\theta$ - $\alpha$ to the last two hypotheses.__

In [19]:
# these models were all fit with 250 simmaps, a phylogeny of 1301 plant species including ErM states
# these models were fit using the following template

```R
OUwie::hOUwie(phy = phylogeny,
              data = data,
              rate.cat = {2 if null_model else 1},
              discrete_model = discrete_model,
              continuous_model = continuous_model,
              nSim = 250,
              null.model = {TRUE if null_model else FALSE},
              lb_discrete_model = lb_discrete_model, # 1e-15
              ub_discrete_model = ub_discrete_model, # 10.000
              lb_continuous_model = c(lb_continuous_model.alpha, lb_continuous_model.sigma_sq, lb_continuous_model.theta), # these were set to defaults
              ub_continuous_model = c(ub_continuous_model.alpha, ub_continuous_model.sigma_sq, ub_continuous_model.theta) # these were set to defaults
)
```

In [16]:
rd_cd <- read_all_rds(dirpath = "../../data/chapter2/rdata/parallel/hie-general3/F00679/", cd = TRUE, strip = "_F00679_CD_250")
rd_cid <- read_all_rds(dirpath = "../../data/chapter2/rdata/parallel/hie-general3/F00679/", cd = FALSE, strip = "_F00679_CID_250")

In [24]:
merge(OUwie::getModelTable(rd_cd, type = "AICc")[c("AICc", "dAICc")],
      OUwie::getModelTable(rd_cid, type = "AICc")[c("AICc", "dAICc")],
      by = 0, all = TRUE, suffixes = c("_CD", "_CID"))

Row.names,AICc_CD,dAICc_CD,AICc_CID,dAICc_CID
<chr>,<dbl>,<dbl>,<dbl>,<dbl>
ARDOUM,156566.78,129654.769,221312.34,208582.4252
ARDOUMA,225011.14,198099.123,46352.32,33622.4039
ARDOUMV,1021122.13,994210.116,241934.92,229204.9964
ARDOUMVA,4797144.25,4770232.240,12729.92,0.0000
EROUM,392226.26,365314.252,721766.58,709036.6637
EROUMA,600719.97,573807.953,18741.80,6011.8794
EROUMV,157222.52,130310.511,346898.80,334168.8801
EROUMVA,39762.09,12850.079,13224.63,494.7128
SYMOUM,296939.01,270026.993,346720.21,333990.2903


In [27]:
# issue with the current implementation - comes from getModelTable() line 739 - error check for number of models should go before the row mismatch error check
# in call to getModelTable inside getModelAvgParams at line 787 (after dropping the failed? models)
rd_cd_avgs <- OUwie::getModelAvgParams(rd_cd, type = "AICc", force = FALSE)

ERROR: Error in if (var(unlist(lapply(model.list, function(x) dim(x$data)[1]))) != : missing value where TRUE/FALSE needed


In [28]:
rd_cd_avgs <- getModelAvgParams_patched(rd_cd, type = "AICc", force = FALSE)

In [29]:
rd_cd_avgs

,rates,alpha,sigma.sq,theta,expected_mean,expected_var,tip_state
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
Ratibida_pinnata,0.001396594,3.188131e-02,0.1729462,-0.8745306,-0.8745309,1.073704e+10,AM
Heliopsis_helianthoides,0.001396594,3.188131e-02,0.1729462,-0.8745306,-0.8745389,5.181933e+09,AM
Liatris_aspera,0.001396594,3.188131e-02,0.1729462,-0.8745306,-0.8745442,3.043808e+09,AM
Arnica_sororia,0.001396594,3.188131e-02,0.1729462,-0.8745306,-0.8745384,5.585084e+09,AM
Arnica_fulgens,0.001396594,3.188131e-02,0.1729462,-0.8745306,-0.8745384,5.585084e+09,AM
Hymenoxys_richardsonii,0.001396594,3.188131e-02,0.1729462,-0.8745306,-0.8745312,9.059580e+09,AM
Helianthus_agrestis,0.001396594,3.188131e-02,0.1729462,-0.8745306,-0.8745312,3.627737e+10,AM
Helianthus_grosseserratus,0.001396594,3.188131e-02,0.1729462,-0.8745306,-0.8745565,6.779518e+08,AM
Helianthus_giganteus,0.001396594,3.188131e-02,0.1729462,-0.8745306,-0.8745565,6.779518e+08,AM
